In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_commodity = pd.read_csv('../data/raw/agmarknet/commodity_prices.csv')
df_cibil     = pd.read_excel('../data/raw/rbi/External_Cibil_Dataset.xlsx')
df_bank      = pd.read_excel('../data/raw/rbi/Internal_Bank_Dataset.xlsx')
df_credit    = pd.read_csv('../data/raw/lending_club/credit_risk.csv')
df_gst_X     = pd.read_csv('../data/raw/gst/X_Train_Data_Input.csv')
df_gst_y     = pd.read_csv('../data/raw/gst/Y_Train_Data_Target.csv')
df_goods     = pd.read_csv('../data/raw/gst/Goods.csv', encoding='latin-1')

print("✅ All datasets reloaded successfully")

✅ All datasets reloaded successfully


In [8]:
# ── CREDIT RISK CLEANING ──────────────────────────────────────────
print("Before cleaning:", df_credit.shape)
print("Nulls per column:\n", df_credit.isnull().sum())

# Step 1: Drop rows where target variable is null
df_credit = df_credit.dropna(subset=['loan_status'])

# Step 2: Fill numeric nulls with median
num_cols = df_credit.select_dtypes(include=[np.number]).columns
df_credit[num_cols] = df_credit[num_cols].fillna(df_credit[num_cols].median())

# Step 3: Fill categorical nulls with mode
cat_cols = df_credit.select_dtypes(include=['object']).columns
for col in cat_cols:
    df_credit[col] = df_credit[col].fillna(df_credit[col].mode()[0])

print("\nAfter cleaning:", df_credit.shape)
print("Remaining nulls:", df_credit.isnull().sum().sum())

Before cleaning: (32581, 12)
Nulls per column:
 person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

After cleaning: (32581, 12)
Remaining nulls: 0


In [9]:
# ── GST DATA CLEANING ─────────────────────────────────────────────
print("Before cleaning:", df_gst_X.shape)
print("\nNull counts per column:")
print(df_gst_X.isnull().sum().sort_values(ascending=False).head(10))

# Step 1: Drop columns where more than 50% values are missing
threshold = 0.5
missing_pct = df_gst_X.isnull().mean()
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print(f"\nDropping {len(cols_to_drop)} columns with >50% nulls: {cols_to_drop}")
df_gst_X = df_gst_X.drop(columns=cols_to_drop)

# Step 2: Fill remaining nulls with median
num_cols_gst = df_gst_X.select_dtypes(include=[np.number]).columns
df_gst_X[num_cols_gst] = df_gst_X[num_cols_gst].fillna(df_gst_X[num_cols_gst].median())

print("\nAfter cleaning:", df_gst_X.shape)
print("Remaining nulls:", df_gst_X.isnull().sum().sum())

Before cleaning: (785133, 23)

Null counts per column:
Column9     732137
Column14    365703
Column5     167180
Column4     127710
Column3     126303
Column15     16456
Column6       3850
Column8       3850
Column0          9
ID               0
dtype: int64

Dropping 1 columns with >50% nulls: ['Column9']

After cleaning: (785133, 22)
Remaining nulls: 0


In [10]:
# ── COMMODITY PRICES CLEANING ─────────────────────────────────────
print("Before:", df_commodity.shape)
print(df_commodity.dtypes)

# Step 1: Parse date column properly
df_commodity['Arrival_Date'] = pd.to_datetime(
    df_commodity['Arrival_Date'], dayfirst=True
)

# Step 2: Rename columns for cleaner access
df_commodity.columns = [
    'state', 'district', 'market', 'commodity',
    'variety', 'grade', 'date',
    'min_price', 'max_price', 'modal_price'
]

# Step 3: Sort by date
df_commodity = df_commodity.sort_values('date').reset_index(drop=True)

# Step 4: Remove price outliers using IQR
Q1 = df_commodity['modal_price'].quantile(0.25)
Q3 = df_commodity['modal_price'].quantile(0.75)
IQR = Q3 - Q1
df_commodity = df_commodity[
    (df_commodity['modal_price'] >= Q1 - 3*IQR) &
    (df_commodity['modal_price'] <= Q3 + 3*IQR)
]

print("\nAfter:", df_commodity.shape)
print(df_commodity.dtypes)
print(df_commodity.head())

Before: (2733, 10)
State                object
District             object
Market               object
Commodity            object
Variety              object
Grade                object
Arrival_Date         object
Min_x0020_Price       int64
Max_x0020_Price       int64
Modal_x0020_Price     int64
dtype: object

After: (2696, 10)
state                  object
district               object
market                 object
commodity              object
variety                object
grade                  object
date           datetime64[ns]
min_price               int64
max_price               int64
modal_price             int64
dtype: object
     state  district      market             commodity           variety  \
0  Gujarat    Amreli    Damnagar     Coriander(Leaves)         Coriander   
1   Kerala  Kottayam  Ettumanoor  Elephant Yam (Suran)             Other   
2   Kerala  Kottayam  Ettumanoor         Ginger(Green)      Green Ginger   
3   Kerala  Kottayam  Ettumanoor      Mango (Raw-R

In [11]:
# ── MERGE CIBIL + BANK DATA ───────────────────────────────────────
df_merged = pd.merge(df_cibil, df_bank, on='PROSPECTID', how='inner')
print(f"Merged shape: {df_merged.shape}")

# Target variable distribution
print("\nTarget distribution:")
print(df_merged['Approved_Flag'].value_counts())
print(df_merged['Approved_Flag'].value_counts(normalize=True).round(3))

# Quick sanity check
print(f"\nNull values in merged: {df_merged.isnull().sum().sum()}")
print(f"Duplicate rows: {df_merged.duplicated().sum()}")

Merged shape: (51336, 87)

Target distribution:
Approved_Flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
Approved_Flag
P2    0.627
P3    0.145
P4    0.115
P1    0.113
Name: proportion, dtype: float64

Null values in merged: 0
Duplicate rows: 0


In [12]:
# ── SAVE ALL CLEANED DATASETS ─────────────────────────────────────
import os
os.makedirs('../data/processed', exist_ok=True)

# Convert Approved_Flag to binary target
# P2 = healthy (1), P1/P3/P4 = stressed (0)
df_merged['stress_label'] = (df_merged['Approved_Flag'] != 'P2').astype(int)
print("Stress label distribution:")
print(df_merged['stress_label'].value_counts())
print(f"Stress rate: {df_merged['stress_label'].mean():.1%}")

# Save all cleaned files
df_merged.to_csv('../data/processed/credit_merged_clean.csv', index=False)
df_credit.to_csv('../data/processed/credit_risk_clean.csv', index=False)
df_commodity.to_csv('../data/processed/commodity_clean.csv', index=False)
df_gst_X.to_csv('../data/processed/gst_features_clean.csv', index=False)
df_gst_y.to_csv('../data/processed/gst_target.csv', index=False)

print("\n✅ All cleaned datasets saved to /data/processed/")
print("\nFinal shapes:")
print(f"Credit merged:    {df_merged.shape}")
print(f"Credit risk:      {df_credit.shape}")
print(f"Commodity prices: {df_commodity.shape}")
print(f"GST features:     {df_gst_X.shape}")

Stress label distribution:
stress_label
0    32199
1    19137
Name: count, dtype: int64
Stress rate: 37.3%

✅ All cleaned datasets saved to /data/processed/

Final shapes:
Credit merged:    (51336, 88)
Credit risk:      (32581, 12)
Commodity prices: (2696, 10)
GST features:     (785133, 22)
